# Queueing networks for LLM scheduling
Run from a local checkout. This notebook shows synthetic model evidence; hardware results are reported separately. No GPU or model download is required here.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "queueing_lab").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "queueing_lab").is_dir(), "Open this notebook inside the checkout"
sys.path.insert(0, str(ROOT))
from queueing_lab.model import Config, Catalog
catalog = Catalog(Config())
catalog.capacity((0.5, 0.5))

## Compare identical offered arrivals
The pressure policy uses only current queues, finite KV feasibility, and the frozen service catalog. Both arms below use gated cohorts, unlike native continuous admission.

In [ ]:
from queueing_lab.workload import generate
from queueing_lab.simulator import simulate
rate = 0.85 * catalog.capacity((0.5, 0.5))["requests_s"]
trace = generate(rate, 30, 20261021)
results = {policy: simulate(Config(), trace, 30, policy) for policy in ("fcfs", "pressure")}
{policy: r["metrics"] for policy, r in results.items()}

In [ ]:
import matplotlib.pyplot as plt
for policy, result in results.items():
    plt.plot(*zip(*result["backlog"]), label=policy)
plt.xlabel("Seconds"); plt.ylabel("All unfinished requests"); plt.legend(); plt.show()

## A network with actual join dependencies
Servers here are distinct simulated resources. This is not measured parallelism on a single GPU and carries no network stability claim.

In [ ]:
from queueing_lab.network import simulate_network
network = simulate_network(Config(), trace, 30, "pressure")
network["metrics"]

## Read the full evidence
See ../REPORT.md, ../docs/THEORY.md, ../results/screen-v2/REPORT.md and ../results/hardware-v1/REPORT.md. The native hardware control is essential: beating gated FCFS alone is not a serving speedup. Follow ../VALIDATE.md for independent falsification.